# Jour 1 · Chapitre 3 — Scénario : migrer le programme SAS de l'équipe Risque
## Formation Beobank · SAS → Python · Cahier participant

**L'histoire.** La licence SAS de l'équipe Risque n'est pas renouvelée. Marc, analyste senior,
vous confie son programme de suivi des contrats :
*"Ce programme tourne tous les mois depuis 2015. Il faut le même résultat en Python, étape par
étape. Je t'ai mis mon code SAS en commentaire au-dessus de chaque bloc."*

**Comment travailler.** Chaque cellule montre le code **SAS d'origine** en commentaire, puis sa
traduction Python avec des trous `______`. Retrouvez l'équivalent Python de chaque instruction SAS.

📎 **Matière à relire :** `SAS_vs_Python.ipynb` (même dossier) — sections 2 (valeurs manquantes),
5 (formats), 6 (exploration), 7 (tri et filtre), 8 (colonne calculée), 9 (PROC SQL) et 10 (export).

## Préparation (déjà écrite, exécutez-la) — l'équivalent du `LIBNAME` + `PROC IMPORT`

In [ ]:
# ── Imports et chargement ─────────────────────────────────────────────────
# SAS : PROC IMPORT DATAFILE="CTR.csv" OUT=ctr DBMS=DLM; DELIMITER=";"; RUN;
import os
import sqlite3
import numpy as np
import pandas as pd

ctr = pd.read_csv("../../data/CTR.csv", sep=";", na_values=".", encoding="utf-8")  # "." SAS => NaN
print("Table ctr chargée :", ctr.shape)

---
## Étape 1 — Regarder la table

*"Je commence toujours par un PROC PRINT et un PROC CONTENTS."*

In [ ]:
# ── PROC PRINT / PROC CONTENTS ────────────────────────────────────────────
# SAS : PROC PRINT DATA=ctr(OBS=5); RUN;
# Quelle méthode affiche les premières lignes ?
print(ctr.______(5))
# SAS : PROC CONTENTS DATA=ctr; RUN;
# Quelle méthode liste colonnes, types et nombre de valeurs non manquantes ?
ctr.______()

---
## Étape 2 — Contrôler les manquants et les statistiques

*"Mon PROC MEANS NMISS me dit combien de soldes sont vides, puis je regarde la distribution."*

In [ ]:
# ── PROC MEANS ────────────────────────────────────────────────────────────
# SAS : PROC MEANS DATA=ctr NMISS; VAR SLD_CTR; RUN;
# Quelle méthode marque les valeurs manquantes (le "." de SAS) ?
print("Soldes manquants :", ctr["SLD_CTR"].______().sum())        # → 137
# SAS : PROC MEANS DATA=ctr N MEAN STD MIN MAX; VAR SLD_CTR; RUN;
# Quelle méthode donne count, mean, std, min, quartiles, max en une fois ?
ctr["SLD_CTR"].______().round(2)

---
## Étape 3 — Le PROC FORMAT des statuts

*"J'avais un format $STATUT. pour afficher des libellés au lieu des codes."*

In [ ]:
# ── PROC FORMAT + PUT ─────────────────────────────────────────────────────
# SAS : PROC FORMAT; VALUE STATUT 4="Clôturé" 6="Résilié"; RUN;
format_statut = {4: "Clôturé", 6: "Résilié"}              # le format devient un dictionnaire
# SAS : DATA ctr; SET ctr; LIB_STATUT = PUT(COD_ECV_CTR, STATUT.); RUN;
# Quelle méthode applique un dictionnaire de correspondance à chaque valeur d'une colonne ?
ctr["LIB_STATUT"] = ctr["COD_ECV_CTR"].______(format_statut)
# SAS : PROC FREQ DATA=ctr; TABLES LIB_STATUT; RUN;
ctr["LIB_STATUT"].______()                       # → Clôturé 105, Résilié 95

---
## Étape 4 — Les filtres du DATA step

*"Je sortais deux tables : les découverts, et les comptes EUR clôturés."*

In [ ]:
# ── DATA step + WHERE ─────────────────────────────────────────────────────
# SAS : DATA decouverts; SET ctr; WHERE SLD_CTR < 0; RUN;
decouverts = ctr[ctr["SLD_CTR"] ______ 0]
print("Découverts :", len(decouverts))                    # → 13
# SAS : DATA eur_clotures; SET ctr; WHERE COD_DEV = "EUR" AND COD_ECV_CTR = 4; RUN;
# En pandas, "AND" entre deux conditions s'écrit avec quel symbole ? (et n'oubliez pas les parenthèses)
eur_clotures = ctr[(ctr["COD_DEV"] == "EUR") ______ (ctr["COD_ECV_CTR"] == 4)]
print("EUR clôturés :", len(eur_clotures))

---
## Étape 5 — Les colonnes calculées IF / THEN / ELSE

*"Attention : en SAS, un solde manquant est plus petit que tout nombre. En Python, non !"*

In [ ]:
# ── IF THEN ELSE ──────────────────────────────────────────────────────────
# SAS : IF COD_ECV_CTR = 6 THEN EST_RESILIE = "O"; ELSE EST_RESILIE = "N";
# Quelle fonction NumPy est le « SI condition ALORS a SINON b » vectorisé ?
ctr["EST_RESILIE"] = np.______(ctr["COD_ECV_CTR"] == ______, "O", "N")

# SAS : IF SLD_CTR = . THEN SENS = "Inconnu";
#       ELSE IF SLD_CTR < 0 THEN SENS = "Débiteur";
#       ELSE SENS = "Créditeur";
conditions = [ctr["SLD_CTR"].isna(), ctr["SLD_CTR"] < 0]
choix = ["Inconnu", "Débiteur"]
# Quelle fonction NumPy gère PLUSIEURS conditions (comme une suite de IF / ELSE IF) ?
# Le dernier ELSE devient la valeur par défaut : laquelle ?
ctr["SENS"] = np.______(conditions, choix, default="______")
ctr["SENS"].value_counts()                                # → Inconnu 137, Créditeur 50, Débiteur 13

---
## Étape 6 — Trier et remplacer les manquants

In [ ]:
# ── PROC SORT + traitement des manquants ──────────────────────────────────
# SAS : PROC SORT DATA=ctr; BY DESCENDING SLD_CTR; RUN;
# Quelle méthode trie un DataFrame ? DESCENDING = ordre croissant ou non ?
top = ctr.______("SLD_CTR", ascending=______)
print(top[["IDT_AC", "SLD_CTR"]].head(3))
# SAS : IF SLD_CTR = . THEN SLD_CTR = 0;
# Quelle méthode remplace les valeurs manquantes par une valeur donnée ?
ctr["SLD_CTR_0"] = ctr["SLD_CTR"].______(0)

---
## Étape 7 — Le PROC SQL de synthèse

*"Ma synthèse finale était un PROC SQL avec GROUP BY. Tu peux le faire en pandas ET en SQL ?"*

In [ ]:
# ── PROC SQL GROUP BY → pandas ────────────────────────────────────────────
# SAS : PROC SQL; SELECT COD_DEV, COUNT(SLD_CTR), SUM(SLD_CTR) FROM ctr GROUP BY COD_DEV; QUIT;
# Quelle méthode regroupe les lignes ? Quelle méthode applique plusieurs agrégations ?
rapport = ctr.______("COD_DEV")["SLD_CTR"].______(["count", "sum"]).reset_index()
rapport

In [ ]:
# ── PROC SQL GROUP BY → vrai SQL avec sqlite3 ─────────────────────────────
conn = sqlite3.connect(":memory:")                        # une base SQL temporaire en mémoire
# Quelle méthode d'un DataFrame l'écrit comme une table SQL ?
ctr.______("CTR", conn, index=False)
# Quelle fonction pandas exécute une requête SQL et renvoie un DataFrame ?
pd.______("SELECT COD_DEV, COUNT(*) AS nb FROM CTR GROUP BY COD_DEV ORDER BY nb DESC", conn)

---
## 🏁 Étape finale — PROC EXPORT

*"Le résultat partait dans un CSV pour le comité, séparateur point-virgule pour Excel belge."*

In [ ]:
# ── PROC EXPORT ───────────────────────────────────────────────────────────
# SAS : PROC EXPORT DATA=rapport OUTFILE="migration_sas.csv" DBMS=DLM; DELIMITER=";"; RUN;
os.makedirs("../../output", exist_ok=True)                # crée le dossier s'il n'existe pas
# Quelle méthode écrit un DataFrame dans un fichier CSV ? Quel séparateur pour Excel en Belgique ?
rapport.______("../../output/migration_sas.csv", sep="______", index=False)
print(open("../../output/migration_sas.csv", encoding="utf-8").read())

✍️ **Pour Marc :** citez 2 différences de comportement entre SAS et Python que vous avez dû gérer
dans cette migration (indice : étape 5 et étape 6).